In [ ]:
import pandas as pd
import plotly.express as px
import geopandas as gpd
import plotly.graph_objects as go
import json
import numpy as np 
import datetime

In [ ]:
df_cim = pd.read_excel("H:\canc_air\data\CODE CIM.revuCBOct23.csv.xlsx")
df_cim.drop('Unnamed: 3', axis=1, inplace=True)
df_patients = pd.read_csv("H:/canc_air/data/data_octobre_2023/Pseudonymisation_provisoire_geocoded_spatial.csv", sep = ";")
df_patients.drop('Unnamed: 0', axis=1, inplace=True)
df_clinique = pd.read_excel("H:/canc_air/data/data_octobre_2023/pseudonymisation_id_sexe_ddn_loc.xlsx")

In [ ]:
df_clinique.dropna(subset=['pseudo_provisoire'])
df_patients.dropna(subset=['pseudo_provisoire'])

In [ ]:
df_adresse_clinique = df_patients.merge(df_clinique, on='pseudo_provisoire', how='left')
df_patients_new_cim = df_adresse_clinique.merge(df_cim, on='topo_initiale_cim10', how='left')

In [ ]:
df_patients_new_cim = df_patients_new_cim.dropna(subset=["date_naissance"])

current_year = datetime.date.today().year
df_patients_new_cim["annee_naissance"] = df_patients_new_cim.date_naissance.str[:4].astype(int)
df_patients_new_cim["age"]= current_year - df_patients_new_cim["annee_naissance"].astype(int)
df_noNa = df_patients_new_cim.drop(["annee_naissance"], axis=1)

age_interval = [(0,9),(10,19),(20,29),(30,39),(40,49),(50,59),(60,69),(70,79),(80,89),(90,99),(100,150)]
df_noNa["ageInterv"] = pd.cut(df_noNa.age, bins=[interval[0] for interval in age_interval] + [age_interval[-1][1]], labels = ['0-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90-99','100+'])



In [ ]:
df_sein = df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Sein"]
df_gyn = df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Gynéco"]
df_hemato =  df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Hemato"]
df_ophtalmo =  df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Ophtalmo"]
df_uro =  df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Uro"]
df_orl =  df_noNa.loc[df_noNa['Proposition_Clémence_1']== "ORL"]
df_gastro =  df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Gastro"]
df_sarcome=  df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Sarcome"]
df_thorax=  df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Thorax"]
df_dermato = df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Dermato"]
df_neuro = df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Neuro"]
df_endo = df_noNa.loc[df_noNa['Proposition_Clémence_1']== "Endocrino"]
list_df_patho = [df_sein,df_gyn,df_hemato, df_ophtalmo,df_uro,df_orl,df_gastro,df_sarcome,df_thorax,df_dermato,df_neuro,df_endo]
list_df_pyramidage = []

for df in list_df_patho: 
    #print(df.columns)
    df_pyramidage = df.groupby(["ageInterv", "patient_sexe"]).size()
    df_pyramidage = df.pivot_table(index="ageInterv", columns="patient_sexe", values="pseudo_provisoire", aggfunc="count", fill_value=0).reset_index()
    list_df_pyramidage.append(df_pyramidage)

### Visualisation par pathologie - pyramide d'age selon la pathologie

In [ ]:
df_pyramidage = list_df_pyramidage[0]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer du sein à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

In [ ]:
df_pyramidage = list_df_pyramidage[1]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer gyneco à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

In [ ]:
df_pyramidage = list_df_pyramidage[2]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer hemato à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

In [ ]:
df_pyramidage = list_df_pyramidage[3]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer ophtalmo à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

fig.show()

In [ ]:
df_pyramidage = list_df_pyramidage[4]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer uro à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

fig.show()

In [ ]:
df_pyramidage = list_df_pyramidage[5]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer ORL à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

fig.show()

In [ ]:
df_pyramidage = list_df_pyramidage[6]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer gastro à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

fig.show()

In [ ]:
df_pyramidage = list_df_pyramidage[7]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer sarcome à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

fig.show()

In [ ]:
df_pyramidage = list_df_pyramidage[8]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer du thorax à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

fig.show()

In [ ]:
df_pyramidage = list_df_pyramidage[9]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer dermarto à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

fig.show()

In [ ]:
df_pyramidage = list_df_pyramidage[10]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer neuro à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

fig.show()

In [ ]:
df_pyramidage = list_df_pyramidage[11]

y_age = df_pyramidage.ageInterv.sort_values(ascending= True)
x_M = df_pyramidage.M
x_F = df_pyramidage.F * -1

fig = go.Figure()
fig.add_trace(go.Bar(y = y_age, x = x_M,
                    name = 'Male',
                    orientation = 'h'))

fig.add_trace(go.Bar(y = y_age, x = x_F,
                    name = 'Female',
                    orientation = 'h'))
fig.update_layout(title = "Age des patients atteints de cancer endo à l'institut Curie",
                  bargap = 0, bargroupgap=0,
                  xaxis = dict(tickvals = [-15000,-10000,-5000,0,5000,10000,15000],
                               ticktext = ['15000', '10000' , '5000', '0', '5000', '10000', '15000'])
                 )

fig.show()

### Visualisation par age - séparation adulte et enfant 

In [ ]:
df_adultes = df_noNa.loc[df_noNa['age'] >= 18]
df_enfants = df_noNa.loc[df_noNa['age'] < 18]
df_grouped_adultes = df_adultes["Proposition_Clémence_1"].value_counts().reset_index()
df_grouped_enfants = df_enfants["Proposition_Clémence_1"].value_counts().reset_index()


In [ ]:
fig = px.histogram(df_grouped_adultes, x= "Proposition_Clémence_1", y = "count", hover_data=["count"]).update_xaxes(categoryorder = "total descending")
fig.update_layout(title = "Répartition des pathologies majoritaires pour nos patients adultes ")
fig.update_layout(hovermode="y")
fig.update_yaxes(title = "Nombre de patients")
fig.update_xaxes(title = "Pathologies majoritaires")
fig.show()

In [ ]:
fig = px.histogram(df_grouped_enfants, x= "Proposition_Clémence_1", y = "count", hover_data=["count"]).update_xaxes(categoryorder = "total descending")
fig.update_layout(title = "Répartition des pathologies majoritaires pour nos patients adultes ")
fig.update_layout(hovermode="y")
fig.update_yaxes(title = "Nombre de patients")
fig.update_xaxes(title = "Pathologies majoritaires")
fig.show()

##### Densité des patients adultes et enfants 

In [ ]:
fig = px.density_mapbox(df_enfants, lat='y', lon='x', #z='patient_count', 
                        radius=5,
                        center=dict(lat=48.8566, lon=2.3522), # Center on Paris
                        zoom=5
                        )

fig.update_layout(
    width=1400,  # Width of the figure
    height=800,  # Height of the figure
    mapbox=dict(
        #zoom=10,
        style="carto-positron",  # Map style (you can also use "open-street-map", "carto-positron", etc.)
    ),
    margin={"r":150, "t":30, "l":30, "b":30},  # Remove margins for a cleaner look
    coloraxis_colorbar=dict(
        title='Number of Patients',
        tickvals=[0, 15000, 30000, 45000, 60000],
        ticktext=['0', '15k', '30k', '45k', '60k']
    )
)


# Display the figure (this won't work in this environment, but the code is provided for reference)
fig.show()

In [ ]:
fig = px.density_mapbox(df_adultes, lat='y', lon='x', #z='patient_count', 
                        radius=5,
                        center=dict(lat=48.8566, lon=2.3522), # Center on Paris
                        zoom=5,
                        opacity=0.5
                        )

fig.update_layout(
    width=1400,  # Width of the figure
    height=800,  # Height of the figure
    mapbox=dict(
        #zoom=10,
        style="carto-positron",  # Map style (you can also use "open-street-map", "carto-positron", etc.)
    ),
    margin={"r":150, "t":30, "l":30, "b":30},  # Remove margins for a cleaner look
    coloraxis_colorbar=dict(
        title='Number of Patients')
    #    tickvals=[0, 15000, 30000, 45000, 60000],
    #    ticktext=['0', '15k', '30k', '45k', '60k']
    #)
)


# Display the figure (this won't work in this environment, but the code is provided for reference)
fig.show()
fig.write_html("H:/canc_air/data/heatmap_figure_adultes.html")
